# A/B testing for business goals, in Python

The farm's test from the slides, worked through in code. The farm is fictional and its 5,600 visitors are simulated for class; every number below matches the slides. Run each cell with Shift + Enter.

In [1]:
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

## 1. The data: one row per visitor

In [2]:
farm = pd.read_csv("https://arizvi.com/tulane/farm.csv")
farm["gave"] = farm["gave"] == "Yes"      # Yes and No become True and False, so Python can count donors
farm.head()

,visitor,date,page,gave,type,gift
0,1,2025-12-02,B,False,NaN,NaN
1,2,2025-12-02,A,False,NaN,NaN
2,3,2025-12-02,A,True,Monthly,25.0
3,4,2025-12-03,B,True,One-time,100.0
4,5,2025-12-03,A,False,NaN,NaN


In [3]:
farm["page"].value_counts().sort_index()

page
A    2800
B    2800
Name: count, dtype: int64

## 2. The three numbers

Lifetime value uses the slides' assumptions: a monthly donor gives for 22 months, and a one-time donor gives 1.3 times.

In [4]:
def add_value(df, months=22):
    """Each visitor's lifetime value: the gift times how often it comes, or 0 if no gift."""
    times = df["type"].map({"Monthly": months, "One-time": 1.3})
    return df.assign(value=(df["gift"] * times).fillna(0))

def three_numbers(df):
    t = df.groupby("page").agg(visitors=("gave", "size"), donors=("gave", "sum"), value=("value", "sum"))
    t["conversion_rate"] = t["donors"] / t["visitors"]
    t["lifetime_value"] = t["value"] / t["donors"]
    t["value_per_visitor"] = t["value"] / t["visitors"]
    return t

farm = add_value(farm)
t = three_numbers(farm)
t.round(2)

,visitors,donors,value,conversion_rate,lifetime_value,value_per_visitor
page,,,,,,
A,2800,224,79520.0,0.08,355.0,28.4
B,2800,336,58800.0,0.12,175.0,21.0


## 3. Could the gap in conversion be chance?

The same four steps as the slide: if the two pages were the same, how big a gap would chance alone make?

In [5]:
shared = t["donors"].sum() / t["visitors"].sum()          # 1. one rate for both pages
se = (shared * (1 - shared) * (1 / t.loc["A", "visitors"] + 1 / t.loc["B", "visitors"])) ** 0.5   # 2. standard error
gap = t.loc["B", "conversion_rate"] - t.loc["A", "conversion_rate"]
z = gap / se                                               # 3. the gap in standard errors
p = 2 * stats.norm.sf(abs(z))                              # 4. how often chance makes a gap this big
print(f"shared rate {shared:.2f}, standard error {se:.4f}, gap {gap:.2f}, z {z:.2f}, p {p:.1e}")

shared rate 0.10, standard error 0.0080, gap 0.04, z 4.99, p 6.1e-07


In [6]:
z, p = proportions_ztest(t["donors"], t["visitors"])      # the same test in one line (page A first, so z is negative)
print(f"z {z:.2f}, p {p:.1e}")

z -4.99, p 6.1e-07


## 4. Could the gap in value per visitor be chance?

Every visitor is worth $550, $130, or nothing. Value per visitor is the average of that column, so the same idea works on the averages.

In [7]:
a = farm.loc[farm["page"] == "A", "value"]
b = farm.loc[farm["page"] == "B", "value"]
gap = a.mean() - b.mean()                                  # page A's lead per visitor
se = (a.var() / len(a) + b.var() / len(b)) ** 0.5
z = gap / se
p = 2 * stats.norm.sf(abs(z))
print(f"gap ${gap:.2f}, standard error ${se:.2f}, z {z:.2f}, p {p:.3f}")
print(f"95% confidence interval: ${gap - 1.96 * se:.2f} to ${gap + 1.96 * se:.2f} a visitor")

gap $7.40, standard error $2.54, z 2.91, p 0.004
95% confidence interval: $2.42 to $12.38 a visitor


In [8]:
res = stats.ttest_ind(a, b, equal_var=False)               # Welch's t-test: the same answer in one line
print(f"t {res.statistic:.2f}, p {res.pvalue:.3f}")

t 2.91, p 0.004


## 5. Change one number

How long monthly donors keep giving decides the call. The slides use 22 months. What happens at 12?

In [9]:
three_numbers(add_value(farm, months=12)).round(2)

,visitors,donors,value,conversion_rate,lifetime_value,value_per_visitor
page,,,,,,
A,2800,224,49520.0,0.08,221.07,17.69
B,2800,336,49800.0,0.12,148.21,17.79


In [10]:
pd.DataFrame({f"{m} months": three_numbers(add_value(farm, months=m))["value_per_visitor"]
              for m in [12, 22, 30]}).round(2)

,12 months,22 months,30 months
page,,,
A,17.69,28.4,36.97
B,17.79,21.0,23.57


## 6. This room

The gifts from button 1 on arizvi.com/tulane, read live from the class sheet. Same functions, same steps. With this few people, expect large p-values: chance moves a small test a lot. The z-test also needs a large sample to be reliable. Finding a 4-point gap takes about 880 visitors a page; the farm's 2,800 a page could find a gap of about 2 points.

In [ ]:
CLASS_SHEET = "https://script.google.com/macros/s/AKfycbzeOcoIA7pEmmXaOUefElwE5W7z1XURHbdJNeMCJz4XcjnKTF_ET65VHZCCTcowKMWSfg/exec"

room = pd.read_csv(CLASS_SHEET + "?rows=give")
room["gave"] = room["gave"] == "Yes"
room = add_value(room)
r = three_numbers(room)
r.round(2)

In [ ]:
if r["donors"].sum() < r["visitors"].sum():
    z, p = proportions_ztest(r["donors"], r["visitors"])
    print(f"conversion: z {z:.2f}, p {p:.2f}")
else:
    print("conversion: everyone gave on both pages, so there is no gap to test")
res = stats.ttest_ind(room.loc[room["page"] == "A", "value"], room.loc[room["page"] == "B", "value"], equal_var=False)
print(f"value per visitor: t {res.statistic:.2f}, p {res.pvalue:.2f}")

## For questions: the same tests in R

```r
farm <- read.csv("https://arizvi.com/tulane/farm.csv")
prop.test(c(224, 336), c(2800, 2800), correct = FALSE)   # X-squared 24.9 (z squared), p 6.1e-07
farm$value <- ifelse(farm$type == "Monthly", farm$gift * 22,
                     ifelse(farm$type == "One-time", farm$gift * 1.3, 0))
t.test(value ~ page, data = farm)                          # Welch's t-test, p 0.004
```

## Try it with AI

Open Gemini in Colab and ask: *Using the DataFrame farm, work out conversion rate, lifetime value, and value per visitor for each page. A monthly gift lasts 22 months, and a one-time gift counts 1.3 times.* Then check its table against step 2. If the numbers differ, find the line that makes the difference.